# Game Phase Analysis — Where Are Games Being Decided?
**Project 1500**

All statistics computed fresh from game files on every run.

We split losses by the move on which the game was effectively decided, defined as
the last move before a terminal outcome (checkmate, resignation, timeout).
This tells us whether to focus on openings, middlegame, or endgame.

In [ ]:
import chess
import chess.pgn
import json
import glob
import io
import re
from collections import Counter, defaultdict
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

USERNAME = 'jf4bes'
EXCLUDE  = {'2025_01', '2025_02', '2025_03'}

# Phase boundaries (half-moves)
OPENING_END    = 20   # moves 1-10
MIDDLEGAME_END = 50   # moves 11-25
# endgame = anything after move 25

## Step 1 — Load all rapid games (both colors, April 2025+)

In [ ]:
def load_rapid_games(username, games_dir='games', exclude=EXCLUDE):
    files = sorted(
        glob.glob(f'{games_dir}/2025_*.json') +
        glob.glob(f'{games_dir}/2026_*.json')
    )
    files = [f for f in files if not any(ex in f for ex in exclude)]
    games = []
    for f in files:
        with open(f, encoding='utf-8') as fh:
            month = json.load(fh)
        for g in month:
            if g.get('time_class') != 'rapid': continue
            white = g.get('white', {})
            black = g.get('black', {})
            if white.get('username','').lower() == username.lower():
                color, my, opp = 'white', white, black
            elif black.get('username','').lower() == username.lower():
                color, my, opp = 'black', black, white
            else:
                continue
            result = my.get('result', '')
            if   result == 'win': outcome = 'win'
            elif result in ('checkmated','timeout','resigned','lose','abandoned'): outcome = 'loss'
            elif result in ('agreed','stalemate','repetition','insufficient','timevsinsufficient','50move'): outcome = 'draw'
            else: continue
            eco_url = ''
            m = re.search(r'\[ECOUrl "([^"]+)"\]', g.get('pgn', ''))
            if m: eco_url = m.group(1)
            games.append({
                'outcome':    outcome,
                'color':      color,
                'end_reason': result,
                'pgn':        g.get('pgn', ''),
                'eco_url':    eco_url,
                'my_rating':  my.get('rating', 0),
                'opp_rating': opp.get('rating', 0),
            })
    return games

games = load_rapid_games(USERNAME)
losses = [g for g in games if g['outcome'] == 'loss']
print(f'Total rapid games:  {len(games)}')
print(f'Losses:             {len(losses)}')
print(f'Win rate:           {100*sum(1 for g in games if g["outcome"]=="win")/len(games):.1f}%')

## Step 2 — Count moves per game and assign phase

**Phase definition (by full moves):**
- Opening: move 1–10 (half-moves 1–20)
- Middlegame: move 11–25 (half-moves 21–50)
- Endgame: move 26+ (half-moves 51+)

The phase is assigned based on the total game length — a proxy for when the game was decided.

In [ ]:
def count_halfmoves(pgn_str):
    """Return number of half-moves played in the game."""
    try:
        game = chess.pgn.read_game(io.StringIO(pgn_str))
        return len(list(game.mainline_moves()))
    except Exception:
        return 0

def assign_phase(halfmoves):
    if halfmoves <= OPENING_END:    return 'Opening (≤ move 10)'
    if halfmoves <= MIDDLEGAME_END: return 'Middlegame (move 11–25)'
    return 'Endgame (move 26+)'

for g in games:
    g['halfmoves'] = count_halfmoves(g['pgn'])
    g['phase']     = assign_phase(g['halfmoves'])

losses = [g for g in games if g['outcome'] == 'loss']

phase_counts = Counter(g['phase'] for g in losses)
total_losses = len(losses)

print(f'Losses by phase (game length proxy):')
for phase in ['Opening (≤ move 10)', 'Middlegame (move 11–25)', 'Endgame (move 26+)']:
    n   = phase_counts[phase]
    pct = 100 * n / total_losses
    print(f'  {phase:30s}: {n:4d} losses  ({pct:.1f}%)')

## Step 3 — Loss distribution by game length

In [ ]:
loss_lengths = [g['halfmoves'] for g in losses]
win_lengths  = [g['halfmoves'] for g in games if g['outcome'] == 'win']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: histogram of loss game lengths
axes[0].hist(loss_lengths, bins=40, color='#c0392b', alpha=0.8, edgecolor='none')
axes[0].axvline(OPENING_END,    color='yellow',      linestyle='--', linewidth=1.5, label='Move 10')
axes[0].axvline(MIDDLEGAME_END, color='lightyellow',  linestyle='--', linewidth=1.5, label='Move 25')
axes[0].set_xlabel('Half-moves played')
axes[0].set_ylabel('Losses')
axes[0].set_title('Distribution of LOSSES by game length')
axes[0].legend()

# Right: wins vs losses overlaid
bins = range(0, max(loss_lengths + win_lengths) + 5, 3)
axes[1].hist(win_lengths,  bins=bins, color='#27ae60', alpha=0.6, label='Wins',   edgecolor='none')
axes[1].hist(loss_lengths, bins=bins, color='#c0392b', alpha=0.6, label='Losses', edgecolor='none')
axes[1].set_xlabel('Half-moves played')
axes[1].set_ylabel('Games')
axes[1].set_title('Wins vs Losses by game length')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Median loss game length: {np.median(loss_lengths):.0f} half-moves ({np.median(loss_lengths)/2:.0f} full moves)')
print(f'Median win  game length: {np.median(win_lengths):.0f} half-moves ({np.median(win_lengths)/2:.0f} full moves)')

## Step 4 — Win rate by game length bucket

In [ ]:
# Bucket games into 5-move windows and compute win rate in each
bucket_size = 10  # half-moves per bucket (= 5 full moves)
max_hm      = max(g['halfmoves'] for g in games)
buckets     = range(0, max_hm + bucket_size, bucket_size)

bucket_stats = defaultdict(lambda: Counter())
for g in games:
    b = (g['halfmoves'] // bucket_size) * bucket_size
    bucket_stats[b][g['outcome']] += 1

xs, wrs, totals = [], [], []
for b in sorted(bucket_stats):
    c = bucket_stats[b]
    t = sum(c.values())
    if t < 5: continue
    xs.append(b // 2)  # convert to full moves for labelling
    wrs.append(100 * c['win'] / t)
    totals.append(t)

fig, ax = plt.subplots(figsize=(13, 4))
colors = ['#27ae60' if wr >= 50 else '#c0392b' for wr in wrs]
bars   = ax.bar(xs, wrs, width=4, color=colors, edgecolor='none', alpha=0.85)
ax.axhline(50, color='gray', linestyle='--', linewidth=1)
ax.axvline(10, color='yellow',     linestyle=':', linewidth=1.5, label='Opening / Middlegame (move 10)')
ax.axvline(25, color='lightyellow', linestyle=':', linewidth=1.5, label='Middlegame / Endgame (move 25)')
ax.set_xlabel('Full move number (game length bucket, 5-move windows)')
ax.set_ylabel('Win %')
ax.set_title('Win rate by game length — where do you perform best/worst?')
ax.legend()
for bar, wr, n in zip(bars, wrs, totals):
    if n >= 10:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                f'{wr:.0f}%', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

## Step 5 — How do losses end? (by phase)

In [ ]:
def classify_end(reason):
    if reason in ('checkmated',):          return 'Checkmate'
    if reason in ('resigned', 'lose'):     return 'Resignation'
    if reason in ('timeout', 'abandoned'): return 'Timeout'
    return 'Other'

phases = ['Opening (≤ move 10)', 'Middlegame (move 11–25)', 'Endgame (move 26+)']
end_types = ['Checkmate', 'Resignation', 'Timeout', 'Other']
colors_end = ['#c0392b', '#e67e22', '#8e44ad', '#888']

phase_end = {ph: Counter() for ph in phases}
for g in losses:
    phase_end[g['phase']][classify_end(g['end_reason'])] += 1

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, phase in zip(axes, phases):
    c      = phase_end[phase]
    total  = sum(c.values())
    values = [c[et] for et in end_types]
    ax.bar(end_types, values, color=colors_end, edgecolor='none')
    ax.set_title(f'{phase}\n({total} losses)')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=20)
plt.suptitle('How do losses end — by game phase?', fontsize=13)
plt.tight_layout()
plt.show()

## Step 6 — Phase breakdown by color

In [ ]:
for color in ['white', 'black']:
    color_losses = [g for g in losses if g['color'] == color]
    total = len(color_losses)
    print(f'As {color} ({total} losses):')
    for phase in phases:
        n   = sum(1 for g in color_losses if g['phase'] == phase)
        pct = 100 * n / total if total else 0
        print(f'  {phase:30s}: {n:3d}  ({pct:.0f}%)')
    print()

## Step 7 — Summary

In [ ]:
opening_pct    = 100 * phase_counts['Opening (≤ move 10)']    / total_losses
middlegame_pct = 100 * phase_counts['Middlegame (move 11–25)'] / total_losses
endgame_pct    = 100 * phase_counts['Endgame (move 26+)']      / total_losses
dominant_phase = max(phase_counts, key=phase_counts.get)

print('GAME PHASE ANALYSIS — SUMMARY')
print('=' * 50)
print(f'\nTotal losses analyzed: {total_losses}')
print(f'  Opening losses:     {phase_counts["Opening (≤ move 10)"]:4d}  ({opening_pct:.1f}%)')
print(f'  Middlegame losses:  {phase_counts["Middlegame (move 11–25)"]:4d}  ({middlegame_pct:.1f}%)')
print(f'  Endgame losses:     {phase_counts["Endgame (move 26+)"]:4d}  ({endgame_pct:.1f}%)')
print(f'\nDominant phase: {dominant_phase}')
print()
if opening_pct > 35:
    print('>> Opening lessons are high priority — a large share of losses happen early.')
if middlegame_pct > 40:
    print('>> Middlegame is the main problem area — opening lessons alone will not fix this.')
if endgame_pct > 30:
    print('>> Endgame study warranted — significant losses in the late game.')